In [ ]:
!pwd

In [ ]:
import re
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pickle

# Model Frameworks
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

# Suppress Convergence/Optimization Warnings common with genomic dimensions
warnings.filterwarnings("ignore")


# ==========================================
# 1. HELPER FUNCTION: DATA PIPELINE (COLUMNS AS GENES)
# ==========================================
def load_and_clean_data(matrix_path):
    # Load Excel file (Samples are rows, Genes/Target are columns)
    df_raw = pd.read_excel(matrix_path, index_col=0)

    target_col_name = "paper_BRCA_Subtype_PAM50"
    
    if target_col_name not in df_raw.columns:
        # Fallback to the absolute last column if the exact name varies
        target_col_name = df_raw.columns[-1]
        print(f"Warning: Target column not found by name. Using last column: '{target_col_name}'")

    # Extract target vector and remove it from the expression matrix
    y = df_raw[target_col_name]
    matrix = df_raw.drop(columns=[target_col_name])

    # Clean suffixes (_SIG, _TF, _NC) from gene column names
    matrix.columns = [
        re.sub(r"_(SIG|TF|NC)$", "", str(col)) for col in matrix.columns
    ]

    # Average identical genes generated by the cleaning process
    if matrix.columns.duplicated().any():
        matrix = matrix.groupby(level=0, axis=1).mean()

    # Map target assignment: 'Basal'/1 -> 1, 'LumA'/0 -> 0
    y = y.map({"Basal": 1, "LumA": 0, 1: 1, 0: 0})

    # Drop samples missing a valid target label
    valid_idx = y.notna()
    X = matrix[valid_idx]
    y = y[valid_idx].astype(int)

    return X, y


# ==========================================
# 2. RUNTIME DATA PARSING & ALIGNMENT
# ==========================================
print("--- Loading & Cleaning Datasets ---")

# Paths direct to the primary data matrix assets
X_train, y_train = load_and_clean_data(
    "benchmarking_count_table/Training.xlsx"
)

X_val, y_val = load_and_clean_data(
    "benchmarking_count_table/Testing.xlsx"
)

# Enforce strict column structural equality
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)


# ==========================================
# 3. HELPER FUNCTION: METRIC & WEIGHT PLOTTING
# ==========================================
def display_and_plot_results(model_name, y_true, y_pred, y_proba, importances):
    print(f"\n==========================================")
    print(f" MODEL EVALUATION: {model_name}")
    print(f"==========================================")

    # Classification Output
    print(
        classification_report(
            y_true, y_pred, target_names=["0 (LumA) ", "1 (Basal)"]
        )
    )
    print(f"Validation ROC-AUC Score : {roc_auc_score(y_true, y_proba):.4f}")
    print(f"Validation F1-Score      : {f1_score(y_true, y_pred):.4f}\n")

    # Extract Top 20 Predictors
    top_20 = pd.Series(importances, index=X_train.columns).nlargest(20)
    top_20_df = pd.DataFrame(
        {"Gene": top_20.index, "Importance_Weight": top_20.values}
    ).reset_index(drop=True)

    print(f"--- Top 20 Genes by Weight ({model_name}) ---")
    print(top_20_df.to_string())

    # Build Visualization Matrix
    plt.figure(figsize=(10, 4))
    sns.barplot(
        x="Importance_Weight",
        y="Gene",
        data=top_20_df,
        palette="magma",
        hue="Gene",
        legend=False,
    )
    plt.title(f"Top 20 Driver Genes - {model_name}", fontsize=12, weight="bold")
    plt.xlabel("Importance / Model Coefficient Weight")
    plt.ylabel("Genes")
    plt.tight_layout()
    plt.show()


In [ ]:
# ==========================================
# ALGORITHM: RANDOM FOREST CLASSIFIER
# ==========================================
rf_model = RandomForestClassifier(n_estimators=100, random_state=55)
rf_model.fit(X_train, y_train)

display_and_plot_results(
    model_name="Random Forest",
    y_true=y_val,
    y_pred=rf_model.predict(X_val),
    y_proba=rf_model.predict_proba(X_val)[:, 1],
    importances=rf_model.feature_importances_,
)

with open("rf_model.pkl", "wb") as file:
    pickle.dump(rf_model, file)

In [ ]:
top_20_rf = pd.Series(rf_model.feature_importances_, index=X_train.columns).nlargest(20).index.tolist()

In [ ]:
# ==========================================
# ALGORITHM: XGBOOST CLASSIFIER
# ==========================================
# Use scale_pos_weight to offset natural clinical cohort label imbalance
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=55,
    eval_metric="logloss",
)
xgb_model.fit(X_train, y_train)

display_and_plot_results(
    model_name="XGBoost",
    y_true=y_val,
    y_pred=xgb_model.predict(X_val),
    y_proba=xgb_model.predict_proba(X_val)[:, 1],
    importances=xgb_model.feature_importances_,
)

with open("xgb_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)

In [ ]:
top_20_xgb = pd.Series(xgb_model.feature_importances_, index=X_train.columns).nlargest(20).index.tolist()

In [ ]:
# ==========================================
# ALGORITHM: SVM (SMALL SAMPLE OPTIMIZED)
# ==========================================
# Linear kernel yields native coefficients (coef_) while avoiding small-sample overfitting
svm_model = SVC(kernel="linear", class_weight="balanced", probability=True, random_state=55)
svm_model.fit(X_train, y_train)

# Absolute values of weights reveal raw magnitude direction regardless of positive/negative slope
svm_weights = np.abs(svm_model.coef_[0])

display_and_plot_results(
    model_name="Support Vector Machine (Linear)",
    y_true=y_val,
    y_pred=svm_model.predict(X_val),
    y_proba=svm_model.predict_proba(X_val)[:, 1],
    importances=svm_weights,
)

with open("svm_model.pkl", "wb") as file:
    pickle.dump(svm_model, file)

In [ ]:
top_20_svm = pd.Series(np.abs(svm_model.coef_[0]), index=X_train.columns).nlargest(20).index.tolist()

In [ ]:
# ==========================================
# ALGORITHM 2: LOGISTIC REGRESSION (GLM)
# ==========================================
# max_iter is increased to ensure convergence on high-dimensional activity matrices
glm_model = LogisticRegression(max_iter=1000, random_state=55)
glm_model.fit(X_train, y_train)

# For GLMs, the feature importance is derived from the absolute coefficients
glm_importances = np.abs(glm_model.coef_[0])

display_and_plot_results(
    model_name="Generalized Linear Model (Logistic)",
    y_true=y_val,
    y_pred=glm_model.predict(X_val),
    y_proba=glm_model.predict_proba(X_val)[:, 1],
    importances=glm_importances,
)

with open("glm_model.pkl", "wb") as file:
    pickle.dump(glm_model, file)

In [ ]:
# Place this right after running your display_and_plot_results function
top_20_glr = pd.Series(np.abs(glm_model.coef_[0]), index=X_train.columns).nlargest(20).index.tolist()

In [ ]:
top_20_rf = pd.Series(rf_model.feature_importances_, index=X_train.columns).nlargest(50).index.tolist()
top_20_xgb = pd.Series(xgb_model.feature_importances_, index=X_train.columns).nlargest(50).index.tolist()
top_20_svm = pd.Series(np.abs(svm_model.coef_[0]), index=X_train.columns).nlargest(50).index.tolist()
top_20_glr = pd.Series(np.abs(glm_model.coef_[0]), index=X_train.columns).nlargest(50).index.tolist()


lncAPNet_GO_df = pd.read_csv("feature_importance.csv")
lncAPNet_GO = lncAPNet_GO_df["Feature"].head(50).tolist()

PASNet_GO_df = pd.read_csv("feature_importance_PASNet_Ablation.csv")
PASNet_GO = PASNet_GO_df["Feature"].head(50).tolist()

lncAPNet_GO_df_scaled = pd.read_csv("feature_importance_scaled.csv")
lncAPNet_GO_scaled = lncAPNet_GO_df_scaled["Feature"].head(50).tolist()

In [ ]:
import numpy as np
import pandas as pd
import gseapy as gp
import stringdb
from sklearn.metrics import roc_auc_score, f1_score

# ==========================================
# 1. LIVE PERFORMANCE METRICS EXTRACTION
# ==========================================

def compute_metrics(model_obj):
    """Dynamically calculates AUC and F1-score using validation data."""
    # Pull the validation datasets explicitly from your workspace
    X_val_data = globals().get('X_val', None)
    y_val_data = globals().get('y_val', None)
    
    if X_val_data is None or y_val_data is None or model_obj is None:
        return "NA", "NA"
    try:
        # Determine continuous confidence scores for AUC
        if hasattr(model_obj, "predict_proba"):
            y_proba = model_obj.predict_proba(X_val_data)[:, 1]
        elif hasattr(model_obj, "decision_function"):
            y_proba = model_obj.decision_function(X_val_data)
        else:
            y_proba = model_obj.predict(X_val_data)
            
        # Determine discrete class labels for F1-score
        y_pred = model_obj.predict(X_val_data)
        
        auc = roc_auc_score(y_val_data, y_proba)
        f1 = f1_score(y_val_data, y_pred)
        return auc, f1
    except Exception:
        return "NA", "NA"

# Fetch your exact live model variables from notebook memory
rf_model_obj = globals().get('rf_model', None)
xgb_model_obj = globals().get('xgb_model', None)
svm_model_obj = globals().get('svm_model', None)
glm_model_obj = globals().get('glm_model', None)

# Dynamically run predictions against validation arrays
rf_auc, rf_f1 = compute_metrics(rf_model_obj)
xgb_auc, xgb_f1 = compute_metrics(xgb_model_obj)
svm_auc, svm_f1 = compute_metrics(svm_model_obj)
glr_auc, glr_f1 = compute_metrics(glm_model_obj)

# Unified performance mapper (hardcoded deep learning vs live ML models)
performance_dict = {
    "lncAPNet_GO": {"AUC": 0.99, "F1": 0.82},
    "PASNet_GO":   {"AUC": 0.99, "F1": 0.89},
    "RF":          {"AUC": rf_auc, "F1": rf_f1},
    "XGB":         {"AUC": xgb_auc, "F1": xgb_f1},
    "SVM":         {"AUC": svm_auc, "F1": svm_f1},
    "GLR":         {"AUC": glr_auc, "F1": glr_f1}
}

# ==========================================
# 2. FEATURE EXTRACTION (TOP 50 GENES)
# ==========================================
X_train_data = globals().get('X_train', None)

def extract_top_genes(model_obj, mode):
    """Safely extracts top 50 features mapping to X_train matrix headers."""
    if model_obj is None or X_train_data is None:
        return []
    try:
        if mode == "importance":
            scores = model_obj.feature_importances_
        elif mode == "coef":
            # Handles coefficient shape flattening safely (handles single vs multi-class arrays)
            if model_obj.coef_.ndim > 1:
                scores = np.abs(model_obj.coef_[0])
            else:
                scores = np.abs(model_obj.coef_)
        return pd.Series(scores, index=X_train_data.columns).nlargest(50).index.tolist()
    except Exception:
        return []

# Extract lists using your established architectures
top_50_rf = extract_top_genes(rf_model_obj, "importance")
top_50_xgb = extract_top_genes(xgb_model_obj, "importance")
top_50_svm = extract_top_genes(svm_model_obj, "coef")
top_50_glr = extract_top_genes(glm_model_obj, "coef")

algorithm_dict = {
    "lncAPNet_GO": globals().get('lncAPNet_GO', []),
    "PASNet_GO": globals().get('PASNet_GO', []),
    "RF": top_50_rf,
    "XGB": top_50_xgb,
    "SVM": top_50_svm,
    "GLR": top_50_glr
}

# ==========================================
# 3. STRING-DB NETWORK PIPELINE
# ==========================================
def fetch_string_metrics(gene_list):
    string_metrics = {"STRING_Edges": "NA"}
    if not gene_list:
        return string_metrics
    try:
        string_ids = stringdb.get_string_ids(gene_list, species=9606)
        if string_ids is None or string_ids.empty:
            return string_metrics
        
        query_items = string_ids['stringId'].tolist()
        network_df = stringdb.get_network(query_items, species=9606)
        if not network_df.empty:
            string_metrics["STRING_Edges"] = len(network_df)
        else:
            string_metrics["STRING_Edges"] = 0
    except Exception:
        pass
    return string_metrics

# ==========================================
# 4. PATHWAY ENRICHMENT ANALYSIS (GSEApy)
# ==========================================
def evaluate_enrichment(gene_list):
    output_metrics = {
        "Sig_GO_BP": "NA", "Sig_KEGG": "NA", "Sig_Reactome": "NA", "Sig_WikiPath": "NA",
        "Min_Adj_P": "NA", "STRING_Edges": "NA"
    }
    if not gene_list:
        return output_metrics
        
    try:
        libraries = [
            'GO_Biological_Process_2023', 
            'KEGG_2021_Human', 
            'Reactome_2022', 
            'WikiPathway_2021_Human'
        ]
        enr = gp.enrichr(gene_list=gene_list, gene_sets=libraries, organism='human', outdir=None)
        res = enr.results
        
        if not res.empty:
            def count_sig_by_db(db_name):
                subset = res[(res['Gene_set'] == db_name) & (res['Adjusted P-value'] < 0.1)]
                return len(subset)

            output_metrics["Sig_GO_BP"] = count_sig_by_db('GO_Biological_Process_2023')
            output_metrics["Sig_KEGG"] = count_sig_by_db('KEGG_2021_Human')
            output_metrics["Sig_Reactome"] = count_sig_by_db('Reactome_2022')
            output_metrics["Sig_WikiPath"] = count_sig_by_db('WikiPathway_2021_Human')
            output_metrics["Min_Adj_P"] = res['Adjusted P-value'].min()
    except Exception:
        pass
        
    string_results = fetch_string_metrics(gene_list)
    output_metrics["STRING_Edges"] = string_results["STRING_Edges"]
    return output_metrics

# ==========================================
# 5. REPORT RUNTIME GENERATOR
# ==========================================
if __name__ == "__main__":
    results_master = {}
    
    for algo_name, gene_list in algorithm_dict.items():
        print(f"Processing evaluation metrics for: {algo_name}...")
        metrics = evaluate_enrichment(gene_list)
        algo_perf = performance_dict.get(algo_name, {"AUC": "NA", "F1": "NA"})
        
        results_master[algo_name] = {
            "Model Performance: AUC": algo_perf["AUC"],
            "Model Performance: F1-score": algo_perf["F1"],
            "Sig GO Biological Process Pathways (FDR < 0.1)": metrics["Sig_GO_BP"],
            "Sig KEGG Pathways (FDR < 0.1)": metrics["Sig_KEGG"],
            "Sig Reactome Pathways (FDR < 0.1)": metrics["Sig_Reactome"],
            "Sig WikiPathways (FDR < 0.1)": metrics["Sig_WikiPath"],
            "Minimum Adjusted P-value (All Libraries)": metrics["Min_Adj_P"],
            "STRING: Total Network Edges": metrics["STRING_Edges"]
        }
        
    df_report = pd.DataFrame(results_master)
    df_report.index.name = "Evaluation Metric"
    df_report.reset_index(inplace=True)
    
    # Custom numerical formatting wrapper 
    def format_full_numbers(val):
        if val == "NA" or val is None:
            return "NA"
        if isinstance(val, (int, float)):
            if val == 0 and not isinstance(val, bool):
                return "0"
            return f"{val:.8f}".rstrip('0').rstrip('.')
        return str(val)

    for col in df_report.columns:
        if col != "Evaluation Metric":
            df_report[col] = df_report[col].apply(format_full_numbers)
    
    print("\n" + "="*115)
    print("                 COMPREHENSIVE BIOLOGICAL & NETWORK EVALUATION REPORT")
    print("="*115)
    print(df_report.to_string(index=False))
    print("="*115)
